In [1]:
!pip install transformers==4.52.4
!pip install bitsandbytes
!pip install sentence-transformers PyPDF2 faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 112.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.2
    Uninstalling tokenizers-0.23.2:
      Successfully uninstalled tokenizers-0.23.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.18.0
    Uninstalling transformers-5.18.0:
      Successfully uninstalled transformers-5.18.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.29.0 requires huggingface-hub<3.0,>=1

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "mistralai/Mistral-7B-v0.1"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_4bit=True,
    device_map="auto"
)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [3]:
def generate_text(prompt, max_new_tokens=100, num_return_sequences=1):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )


    inputs = inputs.to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_return_sequences=num_return_sequences,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.9,
        )

    return [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]

In [4]:
from sentence_transformers import SentenceTransformer
from PyPDF2 import PdfReader
import faiss

In [5]:
def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    full_text = ""

    for page in reader.pages:
        full_text += page.extract_text() + "\n"
    return full_text

In [6]:
def chunk_text(text, chunk_size=500, overlap=50):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

In [7]:
def embed_chunks(chunks, model_name='sentence-transformers/all-MiniLM-L6-v2'):
    model = SentenceTransformer(model_name)
    embeddings = model.encode(chunks, convert_to_numpy=True)
    return model, embeddings

In [8]:
def create_faiss_index(embeddings):
    dim = embeddings.shape[1]                ##  [0.1, 0.2, 0.3,      384]      [0,384]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)
    return index

In [9]:
def search_index(query, model, index, chunks, k=3):
    query_embedding = model.encode([query], convert_to_numpy=True)

    distances, indices = index.search(query_embedding, k)
    return [chunks[i] for i in indices[0]]

In [14]:

file_path = "/content/rag_practice_knowledge_base.txt"

# 1. Read the text file
with open(file_path, "r", encoding="utf-8") as file:
    text = file.read()

# 2. Split the text into chunks
chunks = chunk_text(text, chunk_size=50, overlap=5)

# 3. Generate embeddings
model_embeddings, embeddings = embed_chunks(chunks)

# 4. Create the FAISS index
index = create_faiss_index(embeddings)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [15]:
question = " What is the difference between AI, machine learning, and deep learning?"
top_chunks = search_index(question, model_embeddings, index, chunks, k=3)

for i, chunk in enumerate(top_chunks, 1):
    print(f"\n--- Chunk {i} ---\n{chunk}")


--- Chunk 1 ---
images, planning, and making predictions. AI is an umbrella term. Machine Learning and Deep Learning are subfields of AI, while Generative AI focuses on producing new content such as text, images, audio, or code. Example: A rule-based program that answers a fixed set of questions can be considered an AI

--- Chunk 2 ---
can be considered an AI application, but it does not necessarily learn from data. # 2. Machine Learning Machine Learning (ML) is a way to build systems that learn patterns from data rather than relying only on manually written rules. Common types of machine learning include: - Supervised learning: The

--- Chunk 3 ---
notes. A RAG application should retrieve relevant passages from this document and use them to answer questions. --- # 1. Artificial Intelligence Artificial Intelligence (AI) is the broad field of building computer systems that perform tasks associated with human intelligence. Examples include understanding language, recognizing images, pla

In [16]:
chunk = top_chunks[0]

prompt = f"Answer the next question: {question} by reading the following text:{chunk}"

In [17]:
answer = generate_text(prompt, max_new_tokens=700)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [18]:
print(answer[0])

Answer the next question:  What is the difference between AI, machine learning, and deep learning? by reading the following text:images, planning, and making predictions. AI is an umbrella term. Machine Learning and Deep Learning are subfields of AI, while Generative AI focuses on producing new content such as text, images, audio, or code. Example: A rule-based program that answers a fixed set of questions can be considered an AI application, but a humanoid robot that can walk, talk, and perform actions is a more advanced form of AI.

## What is the difference between AI, machine learning, and deep learning?

Answer: The difference between AI, machine learning, and deep learning is:

1. AI is the broadest category, encompassing all technologies that enable computers to think and act like humans.
2. Machine learning is a subset of AI that involves teaching computers to learn from data and recognize patterns without being programmed with specific rules.
3. Deep learning is a subset of ma